# 🧠 Module 1: Environment Setup & First Chat with MiniMind

Welcome to MiniMind! This notebook sets up the environment and lets you chat with a pre-trained model.

**What you'll learn:**
- How to set up a Google Colab environment for LLM inference
- How to load a pre-trained language model using HuggingFace Transformers
- How to generate text with MiniMind using temperature and top-p sampling

> 💡 **Make sure to enable GPU**: Runtime → Change runtime type → T4 GPU

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required dependencies
!pip install -q modelscope transformers torch

## 📁 Clone the Repository

We'll clone the MiniMind-Colab repository which contains the model code, datasets, and training scripts.

In [ ]:
!git clone https://github.com/Boyu-Zhang-UOI/minimind-colab /content/minimind-colab
%cd /content/minimind-colab
!pip install -q -r requirements.txt

## ⚠️ Restart Runtime

After installing the requirements, **you must restart the Colab runtime** so that the
updated packages (especially NumPy ≥ 2.0) are loaded from scratch.  
Run the cell below — it will terminate the current kernel.  
Once the runtime reconnects, continue from the next cell.

In [ ]:
# Restart runtime to ensure updated packages (e.g. numpy>=2.0) are loaded
import os
os.kill(os.getpid(), 9)

In [ ]:
import sys
sys.path.insert(0, '/content/minimind-colab')
print("Python path configured!")

## 🗂️ Directory Structure

```
minimind-colab/
├── model/                  # Model architecture & tokenizer
│   ├── model_minimind.py   # MiniMindConfig + MiniMindForCausalLM
│   └── tokenizer.json      # BPE tokenizer (vocab size: 6400)
├── dataset/                # Dataset classes
│   └── lm_dataset.py       # PretrainDataset, SFTDataset, etc.
├── trainer/                # Training scripts
│   ├── train_pretrain.py   # Pre-training
│   ├── train_full_sft.py   # Supervised fine-tuning
│   └── trainer_utils.py    # init_model, get_lr, setup_seed
├── notebooks/              # ← You are here!
└── requirements.txt
```

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Inference will be slow on CPU.")

## 💾 Optional: Mount Google Drive for Persistent Storage

If you want to save checkpoints and avoid re-downloading on restart, mount your Google Drive.

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Then use paths like:
# # save_dir = '/content/drive/MyDrive/minimind-checkpoints'
print("Google Drive mount skipped (uncomment above to enable)")

## 📥 Download Pre-trained MiniMind-3 Model

We'll download the pre-trained MiniMind-3 model from ModelScope. This is a ~150M parameter model that has been pre-trained on Chinese and English text.

The model will be saved to `/content/minimind-3/`.

In [ ]:
!modelscope download --model gongjy/minimind-3 --local_dir /content/minimind-3
print("✅ Model downloaded!")

## 🤖 Load the Model

We'll use HuggingFace's `AutoModelForCausalLM` to load the MiniMind-3 model. The model is stored in the standard Transformers format.

**Key details:**
- Model is loaded in `float16` (half precision) to save GPU memory
- We use `.eval()` mode since we're doing inference only

In [ ]:
import torch
import sys
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('/content/minimind-3', trust_remote_code=True)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained('/content/minimind-3', trust_remote_code=True)
model = model.half().eval().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded!")
print(f"   Parameters: {total_params/1e6:.1f}M")
print(f"   Device: {device}")
print(f"   Dtype: {next(model.parameters()).dtype}")

## 💬 Chat with MiniMind!

Now let's chat with the model. We'll use `temperature` and `top_p` sampling for diverse responses:

- **Temperature**: Higher values → more creative/random, lower → more focused
- **Top-p (nucleus sampling)**: Only sample from the top-p probability mass

In [ ]:
def chat(prompt, max_new_tokens=512, temperature=0.85, top_p=0.95):
    """Generate a response for a given user prompt."""
    conversation = [{"role": "user", "content": prompt}]
    inputs_text = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(inputs_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(
        generated_ids[0][len(inputs["input_ids"][0]):],
        skip_special_tokens=True
    )
    return response

# Test with several prompts
test_prompts = [
    "为什么天空是蓝色的?",
    "请用Python写一个斐波那契函数",
    "解释什么是机器学习"
]

for prompt in test_prompts:
    print(f"💬 User: {prompt}")
    response = chat(prompt)
    print(f"🧠 MiniMind: {response}")
    print("=" * 50)

## 📝 Student Exercise

Try modifying the `chat` function to experiment with different settings:

1. **Lower temperature** (e.g., `temperature=0.3`): Does the model become more deterministic?
2. **Higher temperature** (e.g., `temperature=1.5`): What happens to response quality?
3. **Different prompts**: Try math questions, coding tasks, or creative writing

```python
# Your experiments here
response = chat("YOUR PROMPT HERE", temperature=0.5, top_p=0.9)
print(response)
```

## 💡 Discussion Questions

1. **Why is the model sometimes wrong?** MiniMind is a small model (~150M params). GPT-4 has ~1.8T params. What capabilities does scale add?

2. **Vocabulary size tradeoff**: MiniMind uses a vocab of 6,400 tokens. Qwen2 uses 151,643. What are the pros and cons of a smaller vocabulary?
   - Smaller vocab → fewer embedding parameters, but more tokens per word
   - Larger vocab → more efficient encoding of common words, but larger embedding matrix

3. **Inference vs Training**: Why do we use `.half()` (float16) for inference but might use bfloat16 for training?

4. **Temperature sampling**: How does temperature change the probability distribution mathematically?
   - `P(token) ∝ exp(logit / temperature)`
   - Temperature → 0: argmax (greedy decoding)
   - Temperature → ∞: uniform random sampling